# Pipeline de Modelado — GFP Implementation Gap

**Descripción:** Entrenamiento y evaluación de modelos de clasificación para predecir la brecha de implementación.
Cubre las tres modalidades: Contrata, Administración Directa (AD) y ARCC.

**Modelos:** Logistic Regression, Lasso, Ridge, Elastic Net, Random Forest, XGBoost  
**Resampling:** Original (O), SMOTE (S), SMOTE-Tomek (ST), Naive Random Sampling (NRS)

**Flujo:**
1. Configuración de modalidad
2. Carga de data procesada
3. Split train/test
4. Resampling
5. Entrenamiento de modelos
6. Evaluación (test + train)
7. Tabla de resultados
8. Curvas ROC
9. Feature importance
10. Exportación

## 0. Configuración — cambiar aquí para cada modalidad

In [1]:
import sys
sys.path.append('C:/15_GFP')  # permite importar desde src/

# ============================================================
# CONFIGURACIÓN PRINCIPAL
# Opciones de MODALIDAD: 'contrata' | 'ad' | 'arcc'
# ============================================================

MODALIDAD    = 'contrata'
RANDOM_STATE = 2023
TEST_SIZE    = 0.2

PATH_DATA    = f'C:/15_GFP/data/processed/{MODALIDAD}/1_data_{MODALIDAD}.xlsx'
DIR_MODELS   = f'C:/15_GFP/outputs/models/{MODALIDAD}'
DIR_RESULTS  = f'C:/15_GFP/outputs/results/{MODALIDAD}'
DIR_FIGURES  = f'C:/15_GFP/outputs/figures/{MODALIDAD}'
DIR_FI       = f'C:/15_GFP/outputs/feature_importance/{MODALIDAD}'

print(f'Modalidad: {MODALIDAD.upper()}')
print(f'Input: {PATH_DATA}')

Modalidad: CONTRATA
Input: C:/15_GFP/data/processed/contrata/1_data_contrata.xlsx


## 1. Importaciones

In [2]:
#pip install xgboost imbalanced-learn

In [3]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.combine import SMOTETomek

# Funciones del proyecto
from src.gfp_utils import (
    evaluate_model,
    build_results_table,
    plot_roc_curves,
    get_feature_importance,
    grid_search_rf,
    grid_search_xgb,
    save_models,
    save_grid_search_results,
)

## 2. Carga de data

In [4]:
data = pd.read_excel(PATH_DATA, engine='openpyxl')

# ⚠️  VERIFICAR ANTES DE CORRER:
# Estas columnas se excluyen porque en versiones anteriores del pipeline no
# habían sido eliminadas en la etapa de limpieza. Si 01_cleaning_pipeline ya
# las excluye correctamente, este drop es redundante pero inofensivo.
# Si deben entrar al modelo como features, comentar las líneas de cols_excluir.
cols_excluir = [
    'avance_fisico_real',
    'porcentaje_ejecucion_financiera',
    'estado_actualizacion_avance',
    'monto_aprobado_soles',
]
data = data.drop(columns=[c for c in cols_excluir if c in data.columns], errors='ignore')

data = data.dropna()

print(f'Registros: {len(data):,} | Variables: {data.shape[1]}')
print(f'\nDistribución brecha_existente:')
print(data['brecha_existente'].value_counts())

Registros: 20,593 | Variables: 238

Distribución brecha_existente:
brecha_existente
1    11022
0     9571
Name: count, dtype: int64


## 3. Split train / test

In [5]:
dep_var   = 'brecha_existente'
pred_vars = [col for col in data.columns if col != dep_var]

X = data[pred_vars]
y = data[dep_var]

x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y
)

print(f'Train: {x_train.shape[0]:,} | Test: {x_test.shape[0]:,}')
print(f'Features: {x_train.shape[1]}')

Train: 16,474 | Test: 4,119
Features: 237


## 4. Resampling

In [6]:
# SMOTE
smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy='all')
x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

# SMOTE-Tomek
smote_tomek = SMOTETomek(random_state=RANDOM_STATE)
x_train_smote_tomek, y_train_smote_tomek = smote_tomek.fit_resample(x_train, y_train)

# Naive Random Sampling
ros = RandomOverSampler(random_state=RANDOM_STATE)
x_train_ros, y_train_ros = ros.fit_resample(x_train, y_train)

print('Distribución tras cada resampling:')
for label, y_ in [('Original', y_train), ('SMOTE', y_train_smote),
                  ('SMOTETomek', y_train_smote_tomek), ('NRS', y_train_ros)]:
    from collections import Counter
    print(f'  {label}: {dict(Counter(y_))}')

Distribución tras cada resampling:
  Original: {1: 8817, 0: 7657}
  SMOTE: {1: 8817, 0: 8817}
  SMOTETomek: {1: 7844, 0: 7844}
  NRS: {1: 8817, 0: 8817}


## 5. Entrenamiento de modelos

### 5.1 Logistic Regression

In [7]:
%%time
lg_model_o   = LogisticRegression(random_state=RANDOM_STATE).fit(x_train, y_train)
lg_model_s   = LogisticRegression(random_state=RANDOM_STATE).fit(x_train_smote, y_train_smote)
lg_model_st  = LogisticRegression(random_state=RANDOM_STATE).fit(x_train_smote_tomek, y_train_smote_tomek)
lg_model_nrs = LogisticRegression(random_state=RANDOM_STATE).fit(x_train_ros, y_train_ros)
print('LG listo.')

LG listo.
CPU times: total: 15.1 s
Wall time: 3.6 s


### 5.2 Lasso (L1)

In [8]:
%%time
lasso_model_o   = LogisticRegressionCV(penalty='l1', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train, y_train)
lasso_model_s   = LogisticRegressionCV(penalty='l1', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train_smote, y_train_smote)
lasso_model_st  = LogisticRegressionCV(penalty='l1', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train_smote_tomek, y_train_smote_tomek)
lasso_model_nrs = LogisticRegressionCV(penalty='l1', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train_ros, y_train_ros)
print('Lasso listo.')

Lasso listo.
CPU times: total: 18min 23s
Wall time: 18min 48s


### 5.3 Ridge (L2)

In [9]:
%%time
ridge_model_o   = LogisticRegressionCV(penalty='l2', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train, y_train)
ridge_model_s   = LogisticRegressionCV(penalty='l2', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train_smote, y_train_smote)
ridge_model_st  = LogisticRegressionCV(penalty='l2', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train_smote_tomek, y_train_smote_tomek)
ridge_model_nrs = LogisticRegressionCV(penalty='l2', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train_ros, y_train_ros)
print('Ridge listo.')

Ridge listo.
CPU times: total: 14min 36s
Wall time: 14min 55s


### 5.4 Elastic Net

In [10]:
%%time
elasticnet_model_o   = LogisticRegressionCV(penalty='elasticnet', solver='saga', l1_ratios=[0.5], cv=10, random_state=RANDOM_STATE).fit(x_train, y_train)
elasticnet_model_s   = LogisticRegressionCV(penalty='elasticnet', solver='saga', l1_ratios=[0.5], cv=10, random_state=RANDOM_STATE).fit(x_train_smote, y_train_smote)
elasticnet_model_st  = LogisticRegressionCV(penalty='elasticnet', solver='saga', l1_ratios=[0.5], cv=10, random_state=RANDOM_STATE).fit(x_train_smote_tomek, y_train_smote_tomek)
elasticnet_model_nrs = LogisticRegressionCV(penalty='elasticnet', solver='saga', l1_ratios=[0.5], cv=10, random_state=RANDOM_STATE).fit(x_train_ros, y_train_ros)
print('Elastic Net listo.')

Elastic Net listo.
CPU times: total: 18min 50s
Wall time: 19min 15s


### 5.5 Random Forest — Grid Search

In [11]:
%%time
rf_search_o   = grid_search_rf(x_train, y_train, random_state=RANDOM_STATE)
rf_search_s   = grid_search_rf(x_train_smote, y_train_smote, random_state=RANDOM_STATE)
rf_search_st  = grid_search_rf(x_train_smote_tomek, y_train_smote_tomek, random_state=RANDOM_STATE)
rf_search_nrs = grid_search_rf(x_train_ros, y_train_ros, random_state=RANDOM_STATE)

for label, s in [('O', rf_search_o), ('S', rf_search_s), ('ST', rf_search_st), ('NRS', rf_search_nrs)]:
    print(f'RF {label}: {s.best_params_}')

RF O: {'max_depth': 20, 'max_features': 60, 'n_estimators': 500}
RF S: {'max_depth': 20, 'max_features': 60, 'n_estimators': 250}
RF ST: {'max_depth': 30, 'max_features': 60, 'n_estimators': 250}
RF NRS: {'max_depth': 20, 'max_features': 90, 'n_estimators': 500}
CPU times: total: 7min 5s
Wall time: 57min


In [12]:
%%time
def _build_rf(search):
    p = search.best_params_
    return RandomForestClassifier(
        max_features=p['max_features'],
        n_estimators=p['n_estimators'],
        max_depth=p['max_depth'],
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

rf_optimal_model_o   = _build_rf(rf_search_o).fit(x_train, y_train)
rf_optimal_model_s   = _build_rf(rf_search_s).fit(x_train_smote, y_train_smote)
rf_optimal_model_st  = _build_rf(rf_search_st).fit(x_train_smote_tomek, y_train_smote_tomek)
rf_optimal_model_nrs = _build_rf(rf_search_nrs).fit(x_train_ros, y_train_ros)
print('RF óptimos entrenados.')

RF óptimos entrenados.
CPU times: total: 3min 1s
Wall time: 26.3 s


### 5.6 XGBoost — Grid Search

In [13]:
%%time
xgb_search_o   = grid_search_xgb(x_train, y_train, random_state=RANDOM_STATE)
xgb_search_s   = grid_search_xgb(x_train_smote, y_train_smote, random_state=RANDOM_STATE)
xgb_search_st  = grid_search_xgb(x_train_smote_tomek, y_train_smote_tomek, random_state=RANDOM_STATE)
xgb_search_nrs = grid_search_xgb(x_train_ros, y_train_ros, random_state=RANDOM_STATE)

for label, s in [('O', xgb_search_o), ('S', xgb_search_s), ('ST', xgb_search_st), ('NRS', xgb_search_nrs)]:
    print(f'XGB {label}: {s.best_params_}')

XGB O: {'colsample_bytree': 0.4, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 250, 'subsample': 0.8}
XGB S: {'colsample_bytree': 0.4, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 250, 'subsample': 1.0}
XGB ST: {'colsample_bytree': 0.4, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 500, 'subsample': 1.0}
XGB NRS: {'colsample_bytree': 0.3, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 500, 'subsample': 0.8}
CPU times: total: 16min 53s
Wall time: 22min 54s


In [14]:
%%time
def _build_xgb(search):
    p = search.best_params_
    return XGBClassifier(
        objective='binary:logistic',
        verbosity=0,
        colsample_bytree=p['colsample_bytree'],
        max_depth=p['max_depth'],
        n_estimators=p['n_estimators'],
        learning_rate=p['learning_rate'],
        subsample=p['subsample'],
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

xgb_optimal_model_o   = _build_xgb(xgb_search_o).fit(x_train, y_train)
xgb_optimal_model_s   = _build_xgb(xgb_search_s).fit(x_train_smote, y_train_smote)
xgb_optimal_model_st  = _build_xgb(xgb_search_st).fit(x_train_smote_tomek, y_train_smote_tomek)
xgb_optimal_model_nrs = _build_xgb(xgb_search_nrs).fit(x_train_ros, y_train_ros)
print('XGB óptimos entrenados.')

XGB óptimos entrenados.
CPU times: total: 1min 20s
Wall time: 7.99 s


## 6. Evaluación

### 6.1 Test set

In [15]:
# Lista de (nombre_modelo, estrategia_resampling, modelo_entrenado)
modelos_test = [
    ('Logistic Regression', 'O',   lg_model_o),
    ('Logistic Regression', 'S',   lg_model_s),
    ('Logistic Regression', 'ST',  lg_model_st),
    ('Logistic Regression', 'NRS', lg_model_nrs),
    ('Lasso',               'O',   lasso_model_o),
    ('Lasso',               'S',   lasso_model_s),
    ('Lasso',               'ST',  lasso_model_st),
    ('Lasso',               'NRS', lasso_model_nrs),
    ('Ridge',               'O',   ridge_model_o),
    ('Ridge',               'S',   ridge_model_s),
    ('Ridge',               'ST',  ridge_model_st),
    ('Ridge',               'NRS', ridge_model_nrs),
    ('Elastic Net',         'O',   elasticnet_model_o),
    ('Elastic Net',         'S',   elasticnet_model_s),
    ('Elastic Net',         'ST',  elasticnet_model_st),
    ('Elastic Net',         'NRS', elasticnet_model_nrs),
    ('Random Forest',       'O',   rf_optimal_model_o),
    ('Random Forest',       'S',   rf_optimal_model_s),
    ('Random Forest',       'ST',  rf_optimal_model_st),
    ('Random Forest',       'NRS', rf_optimal_model_nrs),
    ('Boosted Trees',       'O',   xgb_optimal_model_o),
    ('Boosted Trees',       'S',   xgb_optimal_model_s),
    ('Boosted Trees',       'ST',  xgb_optimal_model_st),
    ('Boosted Trees',       'NRS', xgb_optimal_model_nrs),
]

results_test = [
    {'model': name, 'sampling': samp, 'metrics': evaluate_model(model, x_test, y_test)}
    for name, samp, model in modelos_test
]

tabla_test = build_results_table(results_test)
tabla_test

,Overall_Accuracy,Roc_Auc,Global_F1_Score,Matthews_Corr_Coef,No_Precision,No_Recall,No_F1_Score,Si_Precision,Si_Recall,Si_F1_Score
O. Logistic Regression,0.779,0.864,0.778,0.584,0.704,0.904,0.792,0.889,0.670,0.764
S. Logistic Regression,0.775,0.860,0.774,0.581,0.696,0.915,0.791,0.898,0.654,0.757
ST. Logistic Regression,0.776,0.862,0.775,0.589,0.694,0.927,0.794,0.910,0.646,0.756
NRS. Logistic Regression,0.776,0.860,0.775,0.583,0.699,0.913,0.791,0.897,0.658,0.759
O. Lasso,0.773,0.839,0.773,0.553,0.724,0.827,0.772,0.828,0.727,0.774
S. Lasso,0.765,0.841,0.765,0.549,0.699,0.868,0.774,0.855,0.676,0.755
ST. Lasso,0.757,0.837,0.757,0.536,0.690,0.868,0.769,0.853,0.661,0.745
NRS. Lasso,0.764,0.840,0.763,0.545,0.698,0.865,0.773,0.852,0.675,0.754
O. Ridge,0.774,0.841,0.774,0.556,0.724,0.831,0.774,0.832,0.725,0.775
S. Ridge,0.769,0.843,0.769,0.557,0.703,0.870,0.778,0.858,0.681,0.760


### 6.2 Training set (RF y XGB)

In [16]:
modelos_train = [
    ('Random Forest', 'O',   rf_optimal_model_o,   x_train,             y_train),
    ('Random Forest', 'S',   rf_optimal_model_s,   x_train_smote,       y_train_smote),
    ('Random Forest', 'ST',  rf_optimal_model_st,  x_train_smote_tomek, y_train_smote_tomek),
    ('Random Forest', 'NRS', rf_optimal_model_nrs, x_train_ros,         y_train_ros),
    ('Boosted Trees', 'O',   xgb_optimal_model_o,   x_train,            y_train),
    ('Boosted Trees', 'S',   xgb_optimal_model_s,   x_train_smote,      y_train_smote),
    ('Boosted Trees', 'ST',  xgb_optimal_model_st,  x_train_smote_tomek,y_train_smote_tomek),
    ('Boosted Trees', 'NRS', xgb_optimal_model_nrs, x_train_ros,        y_train_ros),
]

results_train = [
    {'model': name, 'sampling': samp, 'metrics': evaluate_model(model, X_tr, y_tr)}
    for name, samp, model, X_tr, y_tr in modelos_train
]

tabla_train = build_results_table(results_train)
tabla_train

,Overall_Accuracy,Roc_Auc,Global_F1_Score,Matthews_Corr_Coef,No_Precision,No_Recall,No_F1_Score,Si_Precision,Si_Recall,Si_F1_Score
O. Random Forest,0.927,0.988,0.927,0.862,0.869,0.993,0.927,0.993,0.869,0.927
S. Random Forest,0.926,0.988,0.926,0.861,0.874,0.996,0.931,0.995,0.857,0.921
ST. Random Forest,0.986,1.000,0.986,0.973,0.974,1.000,0.986,1.000,0.973,0.986
NRS. Random Forest,0.933,0.991,0.933,0.873,0.884,0.996,0.937,0.996,0.870,0.928
O. Boosted Trees,0.815,0.898,0.815,0.642,0.753,0.895,0.818,0.891,0.745,0.812
S. Boosted Trees,0.824,0.904,0.822,0.660,0.773,0.918,0.839,0.899,0.730,0.806
ST. Boosted Trees,0.873,0.945,0.872,0.752,0.830,0.938,0.880,0.929,0.807,0.864
NRS. Boosted Trees,0.834,0.911,0.832,0.679,0.782,0.925,0.848,0.909,0.742,0.817


## 7. Curvas ROC

In [17]:
# Random Forest — test
plot_roc_curves(
    models_dict={
        'RF O.': rf_optimal_model_o, 'RF S.': rf_optimal_model_s,
        'RF ST.': rf_optimal_model_st, 'RF NRS.': rf_optimal_model_nrs,
    },
    X=x_test, y=y_test,
    title=f'ROC — Random Forest ({MODALIDAD.upper()}) — Test',
    filepath=f'{DIR_FIGURES}/roc_rf_test.jpg',
)

# XGBoost — test
plot_roc_curves(
    models_dict={
        'XGB O.': xgb_optimal_model_o, 'XGB S.': xgb_optimal_model_s,
        'XGB ST.': xgb_optimal_model_st, 'XGB NRS.': xgb_optimal_model_nrs,
    },
    X=x_test, y=y_test,
    title=f'ROC — Boosted Trees ({MODALIDAD.upper()}) — Test',
    filepath=f'{DIR_FIGURES}/roc_xgb_test.jpg',
)

# Random Forest — train
plot_roc_curves(
    models_dict={
        'RF O.': rf_optimal_model_o, 'RF S.': rf_optimal_model_s,
        'RF ST.': rf_optimal_model_st, 'RF NRS.': rf_optimal_model_nrs,
    },
    X=x_train, y=y_train,
    title=f'ROC — Random Forest ({MODALIDAD.upper()}) — Train',
    filepath=f'{DIR_FIGURES}/roc_rf_train.jpg',
)

# XGBoost — train
plot_roc_curves(
    models_dict={
        'XGB O.': xgb_optimal_model_o, 'XGB S.': xgb_optimal_model_s,
        'XGB ST.': xgb_optimal_model_st, 'XGB NRS.': xgb_optimal_model_nrs,
    },
    X=x_train, y=y_train,
    title=f'ROC — Boosted Trees ({MODALIDAD.upper()}) — Train',
    filepath=f'{DIR_FIGURES}/roc_xgb_train.jpg',
)

ROC guardada: C:/15_GFP/outputs/figures/contrata/roc_rf_test.jpg
ROC guardada: C:/15_GFP/outputs/figures/contrata/roc_xgb_test.jpg
ROC guardada: C:/15_GFP/outputs/figures/contrata/roc_rf_train.jpg
ROC guardada: C:/15_GFP/outputs/figures/contrata/roc_xgb_train.jpg


## 8. Feature Importance (RF y XGB)

In [18]:
fi_models = [
    ('rf_o',   rf_optimal_model_o),
    ('rf_s',   rf_optimal_model_s),
    ('rf_st',  rf_optimal_model_st),
    ('rf_nrs', rf_optimal_model_nrs),
    ('xgb_o',   xgb_optimal_model_o),
    ('xgb_s',   xgb_optimal_model_s),
    ('xgb_st',  xgb_optimal_model_st),
    ('xgb_nrs', xgb_optimal_model_nrs),
]

fi_results = {}
for name, model in fi_models:
    fi_results[name] = get_feature_importance(model, pred_vars, top_n=50)

# Preview del mejor (RF original)
fi_results['rf_o'].head(10)

,vars,score
0,n_modificaciones,0.319950
1,log_monto_aprobado,0.174896
2,n_adicionales_obra,0.078150
3,plazo_ejecucion_dias,0.070293
4,anio_inicio_obra,0.046680
5,n_informes_control,0.031597
6,n_obras_relacionadas,0.014550
7,Region_sierra centro/norte,0.011921
8,n_deductivos_obra,0.011840
9,naturaleza_obra_Mejoramiento,0.010463


## 9. Exportación

In [19]:
import os
for d in [DIR_MODELS, DIR_RESULTS, DIR_FIGURES, DIR_FI]:
    os.makedirs(d, exist_ok=True)

# 9.1 Tablas de resultados
tabla_test.to_excel(f'{DIR_RESULTS}/results_test.xlsx')
tabla_train.to_excel(f'{DIR_RESULTS}/results_train.xlsx')
print('Tablas de resultados guardadas.')

# 9.2 Modelos
save_models(
    models_dict={
        'lg_o': lg_model_o, 'lg_s': lg_model_s, 'lg_st': lg_model_st, 'lg_nrs': lg_model_nrs,
        'lasso_o': lasso_model_o, 'lasso_s': lasso_model_s, 'lasso_st': lasso_model_st, 'lasso_nrs': lasso_model_nrs,
        'ridge_o': ridge_model_o, 'ridge_s': ridge_model_s, 'ridge_st': ridge_model_st, 'ridge_nrs': ridge_model_nrs,
        'elasticnet_o': elasticnet_model_o, 'elasticnet_s': elasticnet_model_s,
        'elasticnet_st': elasticnet_model_st, 'elasticnet_nrs': elasticnet_model_nrs,
        'rf_o': rf_optimal_model_o, 'rf_s': rf_optimal_model_s,
        'rf_st': rf_optimal_model_st, 'rf_nrs': rf_optimal_model_nrs,
        'xgb_o': xgb_optimal_model_o, 'xgb_s': xgb_optimal_model_s,
        'xgb_st': xgb_optimal_model_st, 'xgb_nrs': xgb_optimal_model_nrs,
    },
    output_dir=DIR_MODELS,
)

# 9.3 Grid search results
save_grid_search_results(
    searches_dict={
        'gs_rf_o': rf_search_o, 'gs_rf_s': rf_search_s,
        'gs_rf_st': rf_search_st, 'gs_rf_nrs': rf_search_nrs,
        'gs_xgb_o': xgb_search_o, 'gs_xgb_s': xgb_search_s,
        'gs_xgb_st': xgb_search_st, 'gs_xgb_nrs': xgb_search_nrs,
    },
    output_dir=DIR_RESULTS,
)

# 9.4 Feature importance
for name, fi_df in fi_results.items():
    fi_df.to_excel(f'{DIR_FI}/{name}_feature_importance.xlsx', index=False)
print(f'Feature importance guardada en: {DIR_FI}')

Tablas de resultados guardadas.
24 modelos guardados en: C:/15_GFP/outputs/models/contrata
Grid search results guardados en: C:/15_GFP/outputs/results/contrata
Feature importance guardada en: C:/15_GFP/outputs/feature_importance/contrata
